# 03. Participación y abstención



Este cuaderno calcula participación y abstención por elección, año y puesto, a partir de las tablas de votos y del censo electoral.

## Entradas y salidas

- Lee `datos/procesados/votos/` y `datos/procesados/censo/`.
- Escribe resultados en `datos/procesados/participacion/` y la auditoría en `docs/fuentes/auditorias/03_participacion/`.

## Definiciones y cruce

**Participación:** se cuentan todos los votos depositados, incluidos los votos en blanco, nulos y no marcados. Los votos válidos se reportan por separado.

**Censo:** cada cruce requiere el mismo año y la misma elección. No se usan censos de otras elecciones.

Los nombres de puestos pueden variar entre fuentes. Cuando no coincide el identificador, el cuaderno intenta cruzar por código y registra el método usado.

In [1]:
# --- 1. Configuración ----------------------------------------------------
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


def raiz_proyecto(inicio: Path) -> Path:
    for c in (inicio.resolve(), *inicio.resolve().parents):
        if (c / "datos").is_dir() and (c / "notebooks").is_dir():
            return c
    raise FileNotFoundError("Abre el cuaderno dentro de Proy_Electoral_Bca.")


RAIZ = raiz_proyecto(Path.cwd())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.analisis import participacion as par  # noqa: E402

VOTOS = RAIZ / "datos" / "procesados" / "votos"
CENSO = RAIZ / "datos" / "procesados" / "censo"
SALIDA = RAIZ / "datos" / "procesados" / "participacion"
AUDITORIA = (RAIZ / "docs" / "fuentes" / "auditorias" / "03_participacion"
             / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
SALIDA.mkdir(parents=True, exist_ok=True)
AUDITORIA.mkdir(parents=True, exist_ok=True)

VERSION_NB = "03.1.0"
OPCIONES_CSV = {"index": False, "float_format": "%.6f", "lineterminator": "\n",
                "encoding": "utf-8-sig"}

try:
    display
except NameError:
    def display(x):  # noqa: D103
        print(x.to_string(index=False) if hasattr(x, "to_string") else x)


def rel(p: Path) -> str:
    try:
        return Path(p).resolve().relative_to(RAIZ).as_posix()
    except ValueError:
        return Path(p).as_posix()


def escribir_csv(df: pd.DataFrame, ruta: Path) -> Path:
    ruta.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(ruta, **OPCIONES_CSV)
    return ruta


def miles(n) -> str:
    return f"{int(n):,}".replace(",", ".")


print("Proyecto:", RAIZ)
print("Salidas: ", rel(SALIDA))

Proyecto: D:\Users\LAURA PEREZ\Desktop\CIENCIA DE DATOS\Proy_Electoral_Bca
Salidas:  datos/procesados/participacion


## 2. Datos disponibles

In [2]:
def cargar_votos(ruta: Path) -> pd.DataFrame:
    d = pd.read_parquet(ruta) if ruta.suffix == ".parquet" else pd.read_csv(ruta, dtype=str)
    d["votos"] = pd.to_numeric(d["votos"], errors="coerce").fillna(0).astype(int)
    return d


# Se agrupan las rutas por archivo lógico para no leer dos veces el mismo año.
candidatos: dict[str, dict[str, Path]] = {}
for p in sorted(VOTOS.rglob("*.csv")) + sorted(VOTOS.rglob("*.parquet")):
    candidatos.setdefault(str(p.with_suffix("")), {})[p.suffix] = p

tablas_votos = []
for clave, versiones in sorted(candidatos.items()):
    d = None
    for suf in (".parquet", ".csv"):
        if suf in versiones:
            try:
                d = cargar_votos(versiones[suf])
                break
            except ImportError:
                continue
    if d is None or d.empty:
        continue
    tablas_votos.append({
        "archivo": Path(clave).name,
        "anio": str(d["anio"].iloc[0]) if "anio" in d.columns else "?",
        "corporacion": str(d["corporacion"].iloc[0]) if "corporacion" in d.columns else "?",
        "datos": d,
    })

censos = {}
for p in sorted(CENSO.glob("censo_*_barrancabermeja.csv")):
    anio = p.stem.split("_")[1]
    censos[anio] = pd.read_csv(p, dtype=str)

print(f"Tablas de votos: {len(tablas_votos)}")
display(pd.DataFrame([{k: v for k, v in t.items() if k != "datos"} for t in tablas_votos]))
print(f"\nAños de censo disponibles: {', '.join(sorted(censos)) or 'ninguno'}")
if not censos:
    raise FileNotFoundError(
        f"No hay censo en {rel(CENSO)}. Ejecuta primero src/ingesta/censo.py")

Tablas de votos: 12


,archivo,anio,corporacion
0,reg_2022_congreso_camara,2022,CAMARA
1,reg_2022_congreso_senado,2022,SENADO
2,reg_2019_terr_alcalde,2019,ALCALDE
3,reg_2019_terr_asamblea,2019,ASAMBLEA
4,reg_2019_terr_concejo,2019,CONCEJO
5,reg_2019_terr_gobernador,2019,GOBERNADOR
6,reg_2019_terr_jal,2019,JAL
7,reg_2023_terr_alcalde,2023,ALCALDE
8,reg_2023_terr_asamblea,2023,ASAMBLEA
9,reg_2023_terr_concejo,2023,CONCEJO



Años de censo disponibles: 2018, 2019, 2022, 2023, 2026


## 3. Cruce de votos y censo

Cada tabla de votos se cruza con el censo del mismo año y elección. Si no hay censo correspondiente, no se calcula la participación.

In [3]:
resultados, controles_todos, sin_censo = {}, [], []
for t in tablas_votos:
    anio, corp = t["anio"], t["corporacion"]
    etiqueta = f"{corp} {anio}"
    if anio not in censos:
        sin_censo.append({"anio": anio, "corporacion": corp,
                          "motivo": "no hay censo de ese año"})
        continue
    try:
        censo_eleccion, tipo = par.censo_de_la_eleccion(censos[anio], corp)
    except AssertionError as e:
        sin_censo.append({"anio": anio, "corporacion": corp, "motivo": str(e)[:140]})
        continue

    cruce = par.cruzar(t["datos"], censo_eleccion)
    resumen = par.resumen_municipal(cruce)
    resultados[etiqueta] = {"cruce": cruce, "resumen": resumen, "anio": anio,
                            "corporacion": corp, "censo_usado": tipo,
                            "votos": t["datos"]}
    controles_todos.append(par.controles(cruce, etiqueta))

    print(f"\n{etiqueta}  (censo: {tipo})")
    print(f"   cruce: {cruce['cruzaron_por_id_puesto']} por identificador, "
          f"{cruce['cruzaron_por_zona_y_codigo']} rescatados por código, "
          f"{len(cruce['sin_cruce'])} sin cruzar")

if sin_censo:
    print("\nSin censo disponible, no se calcula participación:")
    display(pd.DataFrame(sin_censo))
    escribir_csv(pd.DataFrame(sin_censo), AUDITORIA / "sin_censo.csv")


CAMARA 2022  (censo: CONGRESO DE LA REPUBLICA)
   cruce: 51 por identificador, 4 rescatados por código, 6 sin cruzar

SENADO 2022  (censo: CONGRESO DE LA REPUBLICA)
   cruce: 51 por identificador, 4 rescatados por código, 6 sin cruzar

ALCALDE 2019  (censo: AUTORIDADES TERRITORIALES)
   cruce: 63 por identificador, 0 rescatados por código, 0 sin cruzar

ASAMBLEA 2019  (censo: AUTORIDADES TERRITORIALES)
   cruce: 63 por identificador, 0 rescatados por código, 0 sin cruzar

CONCEJO 2019  (censo: AUTORIDADES TERRITORIALES)
   cruce: 63 por identificador, 0 rescatados por código, 0 sin cruzar

GOBERNADOR 2019  (censo: AUTORIDADES TERRITORIALES)
   cruce: 63 por identificador, 0 rescatados por código, 0 sin cruzar

JAL 2019  (censo: AUTORIDADES TERRITORIALES)
   cruce: 62 por identificador, 0 rescatados por código, 0 sin cruzar

ALCALDE 2023  (censo: AUTORIDADES TERRITORIALES)
   cruce: 66 por identificador, 1 rescatados por código, 0 sin cruzar

ASAMBLEA 2023  (censo: AUTORIDADES TERRITOR

## 4. Participación por elección

In [4]:
filas = []
for etiqueta, r in sorted(resultados.items()):
    filas.append({"anio": r["anio"], "corporacion": r["corporacion"],
                  "censo_usado": r["censo_usado"], **r["resumen"]})
panorama = pd.DataFrame(filas).sort_values(["anio", "corporacion"])
display(panorama[["anio", "corporacion", "cobertura_de_puestos", "potencial_electoral",
                  "votos_depositados", "participacion", "abstencion", "votos_validos",
                  "votos_no_validos", "pct_no_validos_sobre_depositados"]])

parciales = panorama[~panorama["cobertura_completa"]]
if len(parciales):
    print("\nATENCIÓN: estas elecciones no cubren todos los puestos.")
    for r in parciales.to_dict("records"):
        print(f"  {r['corporacion']} {r['anio']}: cruzaron {r['puestos_cruzados']} de "
              f"{r['puestos_en_votos']} puestos ({r['cobertura_de_puestos']} %).")
    print("  La tasa de participación sigue siendo interpretable porque faltan votos y")
    print("  potencial de los mismos puestos, pero el potencial que aparece en la tabla")
    print("  NO es el del municipio completo. No se debe citar como tal.")
escribir_csv(panorama, SALIDA / "participacion_por_eleccion.csv")

print("\nCómo se lee:")
print("  participacion = votos depositados ÷ potencial × 100. Incluye blancos, nulos y")
print("  no marcados, porque todos los depositó alguien que fue a votar.")
print("  votos_no_validos = blancos + nulos + no marcados.")

,anio,corporacion,cobertura_de_puestos,potencial_electoral,votos_depositados,participacion,abstencion,votos_validos,votos_no_validos,pct_no_validos_sobre_depositados
0,2019,ALCALDE,100.0,175303,121980,69.58,30.42,113679,8301,6.81
2,2019,ASAMBLEA,100.0,175303,119968,68.43,31.57,77485,42483,35.41
5,2019,CONCEJO,100.0,175303,121765,69.46,30.54,111489,10276,8.44
7,2019,GOBERNADOR,100.0,175303,120891,68.96,31.04,99632,21259,17.59
9,2019,JAL,100.0,175106,116070,66.29,33.71,71755,44315,38.18
4,2022,CAMARA,90.2,166797,90764,54.42,45.58,78773,11991,13.21
11,2022,SENADO,90.2,166797,91527,54.87,45.13,80691,10836,11.84
1,2023,ALCALDE,100.0,187483,132133,70.48,29.52,123383,8750,6.62
3,2023,ASAMBLEA,100.0,187483,128983,68.80,31.20,88695,40288,31.24
6,2023,CONCEJO,100.0,187483,131961,70.39,29.61,119854,12107,9.17



ATENCIÓN: estas elecciones no cubren todos los puestos.
  CAMARA 2022: cruzaron 55 de 61 puestos (90.2 %).
  SENADO 2022: cruzaron 55 de 61 puestos (90.2 %).
  La tasa de participación sigue siendo interpretable porque faltan votos y
  potencial de los mismos puestos, pero el potencial que aparece en la tabla
  NO es el del municipio completo. No se debe citar como tal.

Cómo se lee:
  participacion = votos depositados ÷ potencial × 100. Incluye blancos, nulos y
  no marcados, porque todos los depositó alguien que fue a votar.
  votos_no_validos = blancos + nulos + no marcados.


## 5. Participación territorial

In [5]:
for etiqueta, r in sorted(resultados.items()):
    t = r["cruce"]["tabla"].sort_values("participacion")
    cols = ["cod_zona", "cod_puesto", "puesto", "potencial_electoral",
            "votos_depositados", "participacion", "abstencion"]
    print(f"\n{'=' * 70}\n{etiqueta} · los 8 puestos de MENOR participación")
    display(t[cols].head(8))
    print(f"{etiqueta} · los 8 de MAYOR participación")
    display(t[cols].tail(8).iloc[::-1])
    escribir_csv(t, SALIDA / f"participacion_por_puesto_"
                             f"{par.canon(r['corporacion']).lower()}_{r['anio']}.csv")


ALCALDE 2019 · los 8 puestos de MENOR participación


,cod_zona,cod_puesto,puesto,potencial_electoral,votos_depositados,participacion,abstencion
10,02,01,IE INFANTAS SEDE MIRAMAR,4030,1815,45.04,54.96
31,04,05,COL EL CASTILLO SEDE D (EL PAL,2725,1328,48.73,51.27
13,02,04,UIS,3240,1729,53.36,46.64
15,02,06,INST TEC SUPERIOR INDUSTRIAL,5048,2710,53.68,46.32
53,98,01,CARCEL,197,109,55.33,44.67
56,99,09,COL. SAN RAFAEL DE CHUCURI,581,346,59.55,40.45
20,03,04,COL LAS BETHLEMITAS,6257,3872,61.88,38.12
57,99,70,ESCUELA CIENAGA DE OPON,288,179,62.15,37.85


ALCALDE 2019 · los 8 de MAYOR participación


,cod_zona,cod_puesto,puesto,potencial_electoral,votos_depositados,participacion,abstencion
9,01,10,COL. DE LAS AMERICAS,135,126,93.33,6.67
60,99,A3,ESC.LAURELES SEDE A,961,896,93.24,6.76
52,07,06,COL. PIER DE FERMAT,280,260,92.86,7.14
40,05,07,INS. TECNICO INDUSTRIAL SEDE H,342,315,92.11,7.89
62,99,B2,EL LLANITO,288,265,92.01,7.99
25,03,09,COL DIEGO HERNANDEZ GALLEGO SD CAMINO S.,423,388,91.73,8.27
41,05,08,INST TECNICO INDUSTRIAL SEDE F,150,137,91.33,8.67
24,03,08,COL JUAN FRANCISCO SARASTI MEGACOLEGIO,494,445,90.08,9.92



ALCALDE 2023 · los 8 puestos de MENOR participación


...

,cod_zona,cod_puesto,puesto,potencial_electoral,votos_depositados,participacion,abstencion
63,99,A5,IE PUEBLO REGAO,226.0,206,91.15,8.85
45,06,04,I E CAMILO TORRES RESTREPO SD F ESCUELA,596.0,516,86.58,13.42
47,06,06,I E CAMILO TORRES RESTREPO SD H ESCUELA,526.0,449,85.36,14.64
54,07,07,C EDUC DE MAGDALENA MEDIO SD F FE Y ALEG,708.0,597,84.32,15.68
46,06,05,I E CAMILO TORRES RESTREPO SD G ESCUELA,750.0,620,82.67,17.33
61,99,A3,ESC.LAURELES SEDE A,1347.0,1105,82.03,17.97
64,99,B1,EL LLANITO SEDE A,1895.0,1494,78.84,21.16
62,99,A4,IE PUEBLO REGAO - SEDE CAMPO 23,474.0,371,78.27,21.73



SENADO 2022 · los 8 puestos de MENOR participación


,cod_zona,cod_puesto,puesto,potencial_electoral,votos_depositados,participacion,abstencion
50,98,01,CARCEL,186.0,54,29.03,70.97
9,02,01,IE INFANTAS SEDE MIRAMAR,4097.0,1431,34.93,65.07
54,99,70,ESCUELA CIENAGA DE OPON,303.0,114,37.62,62.38
29,04,05,COL EL CASTILLO SEDE D EL PALMAR,2579.0,984,38.15,61.85
52,99,07,ESC. MESETA DE SAN RAFAEL SEDE A,646.0,251,38.85,61.15
13,02,05,INST TEC SUPERIOR INDUSTRIAL,4982.0,2107,42.29,57.71
53,99,09,COL. SAN RAFAEL DE CHUCURI,586.0,253,43.17,56.83
12,02,04,UIS,3366.0,1524,45.28,54.72


SENADO 2022 · los 8 de MAYOR participación


,cod_zona,cod_puesto,puesto,potencial_electoral,votos_depositados,participacion,abstencion
31,04,07,COL. EL CASTILLO SEDE B,793.0,575,72.51,27.49
24,03,10,COL. MIGUEL ANTONIO CARO,726.0,503,69.28,30.72
38,05,07,INS. TECNICO INDUSTRIAL SEDE H,446.0,305,68.39,31.61
39,05,08,INST TECNICO INDUSTRIAL SEDE F,179.0,119,66.48,33.52
8,01,09,COL. DE LAS AMERICAS,335.0,216,64.48,35.52
27,04,03,COL TEC INDUSTRIAL SEDE E,4326.0,2741,63.36,36.64
17,03,03,COL TEC DE COMERCIO SEDE C,3565.0,2250,63.11,36.89
49,07,06,COL PIERRE DE FERMAT,733.0,459,62.62,37.38


### 5.1. Participación por comuna

La agregación usa la comuna registrada en las tablas de votos.

In [6]:
for etiqueta, r in sorted(resultados.items()):
    votos = r["votos"]
    col_comuna = "comuna_normalizada" if "comuna_normalizada" in votos.columns else "comuna"
    mapa = (votos[["id_puesto", "cod_comuna", col_comuna]].drop_duplicates("id_puesto"))
    t = r["cruce"]["tabla"].merge(mapa, on="id_puesto", how="left")
    g = (t.groupby(["cod_comuna", col_comuna], as_index=False)
         .agg(puestos=("id_puesto", "nunique"),
              potencial=("potencial_electoral", "sum"),
              depositados=("votos_depositados", "sum")))
    g["participacion"] = (g["depositados"] / g["potencial"] * 100).round(2)
    g["abstencion"] = (100 - g["participacion"]).round(2)
    g = g.sort_values("participacion", ascending=False)
    print(f"\n{etiqueta} · participación por comuna")
    display(g)
    escribir_csv(g, SALIDA / f"participacion_por_comuna_"
                             f"{par.canon(r['corporacion']).lower()}_{r['anio']}.csv")

print("\nADVERTENCIA (prevención de falacia ecológica):")
print("Una comuna con participación baja NO significa que una persona de esa comuna vote")
print("menos. Son datos agregados por puesto de votación, y un puesto no es un grupo")
print("social homogéneo: la gente vota donde está inscrita, no necesariamente donde vive.")


ALCALDE 2019 · participación por comuna


,cod_comuna,comuna_normalizada,puestos,potencial,depositados,participacion,abstencion
9,09,CORREGIMIENTO EL LLANITO,2,2381,1833,76.98,23.02
8,08,CORREGIMIENTO EL CENTRO,3,11305,8617,76.22,23.78
5,05,COMUNA 5,9,25137,18354,73.02,26.98
7,07,COMUNA 7,6,14451,10392,71.91,28.09
6,06,COMUNA 6,4,14426,10323,71.56,28.44
1,01,COMUNA 1,10,28180,19850,70.44,29.56
4,04,COMUNA 4,7,20997,14760,70.30,29.70
3,03,COMUNA 3,10,27509,19330,70.27,29.73
10,10,CORREG. LA FORTUNA AUTOPISTA,1,1765,1147,64.99,35.01
11,11,CORRE. MESETA DE SAN RAFAEL,1,638,404,63.32,36.68



ALCALDE 2023 · participación por comuna


,cod_comuna,comuna_normalizada,puestos,potencial,depositados,participacion,abstencion
9,09,CORREGIMIENTO EL LLANITO,2,3004.0,2345,78.06,21.94
8,08,CORREGIMIENTO EL CENTRO,5,13591.0,10377,76.35,23.65
5,05,COMUNA 5,9,26564.0,19738,74.30,25.70
7,07,COMUNA 7,7,18058.0,12984,71.90,28.10
3,03,COMUNA 3,11,29083.0,20747,71.34,28.66
1,01,COMUNA 1,9,28274.0,20166,71.32,28.68
4,04,COMUNA 4,7,21676.0,15313,70.64,29.36
6,06,COMUNA 6,5,16438.0,11412,69.42,30.58
13,13,CORREG CIENAGA DEL OPON,1,361.0,247,68.42,31.58
12,12,CORREG. SAN RAFAEL DE CHUCURI,1,649.0,428,65.95,34.05



ASAMBLEA 2019 · participación por comuna


...


JAL 2023 · participación por comuna


,cod_comuna,comuna_normalizada,puestos,potencial,depositados,participacion,abstencion
8,09,CORREGIMIENTO EL LLANITO,2,3004.0,2304,76.70,23.30
7,08,CORREGIMIENTO EL CENTRO,5,13591.0,10325,75.97,24.03
4,05,COMUNA 5,9,26564.0,19153,72.10,27.90
6,07,COMUNA 7,7,18058.0,12555,69.53,30.47
3,04,COMUNA 4,7,21676.0,14960,69.02,30.98
2,03,COMUNA 3,11,29083.0,19938,68.56,31.44
0,01,COMUNA 1,9,28274.0,19238,68.04,31.96
5,06,COMUNA 6,5,16438.0,10985,66.83,33.17
11,12,CORREG. SAN RAFAEL DE CHUCURI,1,649.0,429,66.10,33.90
12,13,CORREG CIENAGA DEL OPON,1,361.0,227,62.88,37.12



SENADO 2022 · participación por comuna


,cod_comuna,comuna_normalizada,puestos,potencial,depositados,participacion,abstencion
5,05,COMUNA 5,9,25620.0,14916,58.22,41.78
1,01,COMUNA 1,9,28206.0,16369,58.03,41.97
4,04,COMUNA 4,7,21490.0,12434,57.86,42.14
3,03,COMUNA 3,10,28518.0,16016,56.16,43.84
7,07,COMUNA 7,6,15723.0,8426,53.59,46.41
6,06,COMUNA 6,3,15744.0,8100,51.45,48.55
2,02,COMUNA 2,6,27842.0,13695,49.19,50.81
8,10,CORREG. LA FORTUNA AUTOPISTA,1,1933.0,899,46.51,53.49
10,12,CORREG. SAN RAFAEL DE CHUCURI,1,586.0,253,43.17,56.83
9,11,CORRE. MESETA DE SAN RAFAEL,1,646.0,251,38.85,61.15



ADVERTENCIA (prevención de falacia ecológica):
Una comuna con participación baja NO significa que una persona de esa comuna vote
menos. Son datos agregados por puesto de votación, y un puesto no es un grupo
social homogéneo: la gente vota donde está inscrita, no necesariamente donde vive.


## 6. Edad del censo y participación

Se compara la composición por edad del censo de cada puesto con su nivel de participación.

La asociación se interpreta a nivel de puesto. No permite concluir cómo votan las personas jóvenes.

In [7]:
for etiqueta, r in sorted(resultados.items()):
    perfil = par.perfil_etario(r["cruce"])
    if perfil.empty:
        continue
    corr_joven = perfil["participacion"].corr(perfil["pct_censo_18_a_30"])
    corr_mayor = perfil["participacion"].corr(perfil["pct_censo_mas_de_60"])
    print(f"\n{etiqueta}")
    print(f"   correlación entre participación y % del censo de 18 a 30 años: {corr_joven:+.3f}")
    print(f"   correlación entre participación y % del censo de más de 60:    {corr_mayor:+.3f}")
    escribir_csv(perfil, SALIDA / f"perfil_etario_"
                                  f"{par.canon(r['corporacion']).lower()}_{r['anio']}.csv")

print("\nUna correlación no es una causa, y aquí ni siquiera es una afirmación sobre")
print("personas: es una relación entre características de puestos de votación.")


ALCALDE 2019
   correlación entre participación y % del censo de 18 a 30 años: +0.229
   correlación entre participación y % del censo de más de 60:    -0.146

ALCALDE 2023
   correlación entre participación y % del censo de 18 a 30 años: +0.371
   correlación entre participación y % del censo de más de 60:    -0.073

ASAMBLEA 2019
   correlación entre participación y % del censo de 18 a 30 años: +0.213
   correlación entre participación y % del censo de más de 60:    -0.143

ASAMBLEA 2023
   correlación entre participación y % del censo de 18 a 30 años: +0.329
   correlación entre participación y % del censo de más de 60:    -0.084

CAMARA 2022
   correlación entre participación y % del censo de 18 a 30 años: +0.295
   correlación entre participación y % del censo de más de 60:    -0.042

CONCEJO 2019
   correlación entre participación y % del censo de 18 a 30 años: +0.223
   correlación entre participación y % del censo de más de 60:    -0.146

CONCEJO 2023
   correlación entre part

## 7. Controles

In [8]:
if controles_todos:
    controles = pd.concat(controles_todos, ignore_index=True)
    escribir_csv(controles, AUDITORIA / "controles.csv")
    resumen_ctl = (controles.groupby(["control", "criticidad"])["pasa"]
                   .agg(pasan=lambda s: int((s == True).sum()),  # noqa: E712
                        fallan=lambda s: int((s == False).sum()))  # noqa: E712
                   .reset_index())
    display(resumen_ctl)
    fallidos = controles[controles["pasa"] == False]  # noqa: E712
    if len(fallidos):
        print("\nControles que no pasan:")
        display(fallidos[["tabla", "control", "valor", "esperado", "criticidad"]])
    bloqueos = sorted(set(fallidos[fallidos["criticidad"] == "Alta"]["control"]))
    alertas = sorted(set(fallidos[fallidos["criticidad"] != "Alta"]["control"]))
else:
    controles, bloqueos, alertas = pd.DataFrame(), [], []

,control,criticidad,pasan,fallan
0,ningún puesto tiene más votos que su potencial,Alta,12,0
1,participación dentro de un rango creíble,Media,12,0
2,puestos rescatados por zona y código,Baja,12,0
3,todo puesto tiene potencial mayor que cero,Alta,12,0
4,todos los puestos de votos encontraron su censo,Alta,10,2



Controles que no pasan:


,tabla,control,valor,esperado,criticidad
0,CAMARA 2022,todos los puestos de votos encontraron su censo,55 de 61,iguales,Alta
5,SENADO 2022,todos los puestos de votos encontraron su censo,55 de 61,iguales,Alta


## 8. Hallazgos

In [9]:
hallazgos = []
if len(panorama):
    terr = panorama[panorama["censo_usado"] == "AUTORIDADES TERRITORIALES"]
    if len(terr):
        p = terr.sort_values("participacion")
        hallazgos.append({
            "hallazgo": f"En las elecciones territoriales, la participación en "
                        f"Barrancabermeja va de {p['participacion'].min():.1f} % a "
                        f"{p['participacion'].max():.1f} % según la corporación.",
            "nivel_de_evidencia": "Dato descriptivo",
            "que_no_significa": "No significa que una corporación interese más que otra. "
                                "La diferencia puede venir de tarjetas no marcadas, que se "
                                "cuentan como depósito pero no como voto válido.",
        })
    for etiqueta, r in sorted(resultados.items()):
        t = r["cruce"]["tabla"]
        if t.empty:
            continue
        bajo = t.nsmallest(1, "participacion").iloc[0]
        alto = t.nlargest(1, "participacion").iloc[0]
        hallazgos.append({
            "hallazgo": f"{etiqueta}: entre puestos, la participación va de "
                        f"{bajo['participacion']:.1f} % en {bajo['puesto']} a "
                        f"{alto['participacion']:.1f} % en {alto['puesto']}.",
            "nivel_de_evidencia": "Dato descriptivo",
            "que_no_significa": "No significa que la gente de un sector sea más o menos "
                                "cívica. La gente vota donde está inscrita, no donde vive, "
                                "y hay puestos especiales con poblaciones particulares.",
        })
        break

if hallazgos:
    for h in hallazgos:
        print(f"\n{'─' * 70}")
        print(h["hallazgo"])
        print(f"  Nivel de evidencia: {h['nivel_de_evidencia']}")
        print(f"  Qué NO significa:   {h['que_no_significa']}")
    escribir_csv(pd.DataFrame(hallazgos), SALIDA / "hallazgos_participacion.csv")


──────────────────────────────────────────────────────────────────────
En las elecciones territoriales, la participación en Barrancabermeja va de 66.3 % a 70.5 % según la corporación.
  Nivel de evidencia: Dato descriptivo
  Qué NO significa:   No significa que una corporación interese más que otra. La diferencia puede venir de tarjetas no marcadas, que se cuentan como depósito pero no como voto válido.

──────────────────────────────────────────────────────────────────────
ALCALDE 2019: entre puestos, la participación va de 45.0 % en IE INFANTAS SEDE MIRAMAR a 93.3 % en COL. DE LAS AMERICAS.
  Nivel de evidencia: Dato descriptivo
  Qué NO significa:   No significa que la gente de un sector sea más o menos cívica. La gente vota donde está inscrita, no donde vive, y hay puestos especiales con poblaciones particulares.


## 9. Estado de la etapa

In [10]:
salidas = sorted(p for p in SALIDA.glob("*.csv"))
estado = {
    "cuaderno": "03_participacion",
    "version_cuaderno": VERSION_NB,
    "generado": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "elecciones_con_participacion": sorted(resultados),
    "elecciones_sin_censo": sin_censo,
    "resumen": panorama.to_dict("records") if len(panorama) else [],
    "controles_alta_fallidos": bloqueos,
    "alertas": alertas,
    "salidas": [rel(p) for p in salidas],
    "pendiente": [
        "Censo 2019: la Registraduría respondió 403. Sin él no hay participación de las "
        "territoriales de 2019 y la comparación entre 2019 y 2023 queda abierta.",
        "Geometrías de comuna: sin ellas no hay mapas ni análisis espacial.",
    ],
}
(AUDITORIA / "estado_etapa.json").write_text(
    json.dumps(estado, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print(f"Elecciones con participación calculada: {len(resultados)}")
for e in sorted(resultados):
    print(f"  {e}")
if sin_censo:
    print(f"\nSin censo: {len(sin_censo)}")
print(f"\nSalidas: {len(salidas)} archivos en {rel(SALIDA)}")
print(f"Auditoría: {rel(AUDITORIA)}")

Elecciones con participación calculada: 12
  ALCALDE 2019
  ALCALDE 2023
  ASAMBLEA 2019
  ASAMBLEA 2023
  CAMARA 2022
  CONCEJO 2019
  CONCEJO 2023
  GOBERNADOR 2019
  GOBERNADOR 2023
  JAL 2019
  JAL 2023
  SENADO 2022

Salidas: 38 archivos en datos/procesados/participacion
Auditoría: docs/fuentes/auditorias/03_participacion/20261004_191756_362200
